# Compare reprocessed proteomics outputs

This notebook validates a rerun of the proteomics preprocessing workflow after renaming the raw DIA-NN input files.

Expected rerun inputs:
- `Data/reprocessed_proteomics/MS_files/250808_all_fp_perseus_export.txt`
- `Data/reprocessed_proteomics/MS_reference_files/251002_references_perseus_export.txt`

The notebook will:
1. compare the rerun Perseus exports against the original reference exports,
2. regenerate `X_proteomics.pkl` for the canonical `Base` and `Special` splits,
3. regenerate `X_proteomics_sig.pkl` using the canonical significance filter,
4. compare all regenerated outputs against the canonical repository outputs,
5. save regenerated files and comparison reports under `Data/reprocessed_proteomics/`.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = None
for cand in [cwd, *cwd.parents]:
    if (cand / 'Data').exists() and (cand / 'notebooks' / 'data_processing').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not resolve project root.')

DATA_ROOT = PROJECT_ROOT / 'Data'
RERUN_ROOT = DATA_ROOT / 'reprocessed_proteomics'
ORIGINAL_PRIVATE_DATA_ROOT = Path('/mnt/jh/nas/PhD/Code/Ms_paint/Data')

sys.path.insert(0, str(PROJECT_ROOT / 'notebooks' / 'data_processing'))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('RERUN_ROOT:', RERUN_ROOT)
print('ORIGINAL_PRIVATE_DATA_ROOT:', ORIGINAL_PRIVATE_DATA_ROOT)


In [ ]:
import pandas as pd

from proteomics_reprocessing_utils import (
    build_proteomics_matrix,
    build_reference_matrix,
    build_significant_matrix,
    compare_dataframes,
    compare_tsv_files,
    split_like_reference,
)

NEW_RAW_EXPORT = RERUN_ROOT / 'MS_files' / '250808_all_fp_perseus_export.txt'
NEW_REF_EXPORT = RERUN_ROOT / 'MS_reference_files' / '251002_references_perseus_export.txt'
OLD_RAW_EXPORT = ORIGINAL_PRIVATE_DATA_ROOT / 'MS_files' / '250808_all_fp_perseus_export.txt'
OLD_REF_EXPORT = ORIGINAL_PRIVATE_DATA_ROOT / 'MS_reference_files' / '251002_references_perseus_export.txt'
FASTA_PATH = DATA_ROOT / 'Fasta_files' / 'Ecoli.fasta'

BASE_PROT = DATA_ROOT / 'Base' / 'X_proteomics.pkl'
SPECIAL_PROT = DATA_ROOT / 'Special' / 'X_proteomics.pkl'
BASE_PROT_SIG = DATA_ROOT / 'Base' / 'X_proteomics_sig.pkl'
SPECIAL_PROT_SIG = DATA_ROOT / 'Special' / 'X_proteomics_sig.pkl'
BASE_LABELS = DATA_ROOT / 'Base' / 'y_labels.pkl'
SPECIAL_LABELS = DATA_ROOT / 'Special' / 'y_labels.pkl'

for path in [NEW_RAW_EXPORT, NEW_REF_EXPORT, OLD_RAW_EXPORT, OLD_REF_EXPORT, FASTA_PATH, BASE_PROT, SPECIAL_PROT, BASE_PROT_SIG, SPECIAL_PROT_SIG]:
    print(path, 'exists' if path.exists() else 'MISSING')


In [ ]:
tsv_report = pd.DataFrame([
    compare_tsv_files('all_fp_perseus_export', NEW_RAW_EXPORT, OLD_RAW_EXPORT),
    compare_tsv_files('references_perseus_export', NEW_REF_EXPORT, OLD_REF_EXPORT),
])

tsv_report_path = RERUN_ROOT / 'comparison_reports' / 'perseus_export_comparison.csv'
tsv_report.to_csv(tsv_report_path, index=False)

tsv_report


In [ ]:
base_ref = pd.read_pickle(BASE_PROT)
special_ref = pd.read_pickle(SPECIAL_PROT)
base_labels_ref = pd.read_pickle(BASE_LABELS)
special_labels_ref = pd.read_pickle(SPECIAL_LABELS)

X_full_new, y_full_new = build_proteomics_matrix(FASTA_PATH, NEW_RAW_EXPORT, exclude_labels=[])
X_base_new, X_special_new = split_like_reference(X_full_new, base_ref, special_ref)
y_base_new = y_full_new.reindex(base_ref.index)
y_special_new = y_full_new.reindex(special_ref.index)

OUT_FULL = RERUN_ROOT / 'regenerated' / 'full'
OUT_BASE = RERUN_ROOT / 'regenerated' / 'Base'
OUT_SPECIAL = RERUN_ROOT / 'regenerated' / 'Special'

X_full_new.to_pickle(OUT_FULL / 'X_proteomics.pkl')
y_full_new.to_pickle(OUT_FULL / 'y_labels.pkl')
X_base_new.to_pickle(OUT_BASE / 'X_proteomics.pkl')
y_base_new.to_pickle(OUT_BASE / 'y_labels.pkl')
X_special_new.to_pickle(OUT_SPECIAL / 'X_proteomics.pkl')
y_special_new.to_pickle(OUT_SPECIAL / 'y_labels.pkl')

print('Saved regenerated full/base/special proteomics outputs.')
print('full:', X_full_new.shape)
print('base:', X_base_new.shape)
print('special:', X_special_new.shape)


In [ ]:
prot_report = pd.DataFrame([
    compare_dataframes('Base/X_proteomics.pkl', X_base_new, base_ref),
    compare_dataframes('Special/X_proteomics.pkl', X_special_new, special_ref),
    compare_dataframes('Base/y_labels.pkl', y_base_new.to_frame('Label'), base_labels_ref.to_frame('Label')),
    compare_dataframes('Special/y_labels.pkl', y_special_new.to_frame('Label'), special_labels_ref.to_frame('Label')),
])

prot_report_path = RERUN_ROOT / 'comparison_reports' / 'proteomics_pickle_comparison.csv'
prot_report.to_csv(prot_report_path, index=False)

prot_report


In [ ]:
base_sig_ref = pd.read_pickle(BASE_PROT_SIG)
special_sig_ref = pd.read_pickle(SPECIAL_PROT_SIG)

X_ref_new = build_reference_matrix(FASTA_PATH, NEW_REF_EXPORT)
X_samples_new = pd.concat([X_base_new, X_special_new], axis=0)
X_sig_full_new = build_significant_matrix(X_samples_new, X_ref_new)
X_base_sig_new, X_special_sig_new = split_like_reference(X_sig_full_new, base_sig_ref, special_sig_ref)

X_sig_full_new.to_pickle(OUT_FULL / 'X_proteomics_sig.pkl')
X_base_sig_new.to_pickle(OUT_BASE / 'X_proteomics_sig.pkl')
X_special_sig_new.to_pickle(OUT_SPECIAL / 'X_proteomics_sig.pkl')

print('Saved regenerated significance-filtered outputs.')
print('X_ref:', X_ref_new.shape)
print('sig full:', X_sig_full_new.shape)


In [ ]:
sig_report = pd.DataFrame([
    compare_dataframes('Base/X_proteomics_sig.pkl', X_base_sig_new, base_sig_ref),
    compare_dataframes('Special/X_proteomics_sig.pkl', X_special_sig_new, special_sig_ref),
])

sig_report_path = RERUN_ROOT / 'comparison_reports' / 'significant_proteomics_comparison.csv'
sig_report.to_csv(sig_report_path, index=False)

sig_report


In [ ]:
summary = pd.concat([
    tsv_report.assign(category='perseus_export'),
    prot_report.assign(category='proteomics_pickle'),
    sig_report.assign(category='significant_pickle'),
], ignore_index=True, sort=False)

summary_path = RERUN_ROOT / 'comparison_reports' / 'overall_summary.csv'
summary.to_csv(summary_path, index=False)

summary[['category', 'name', 'exact_equal', 'same_shape', 'same_columns', 'same_index', 'n_diff_cells', 'max_abs_diff']]
